In [ ]:
#Import dependencies
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pmdarima as pm
from pmdarima.arima import ndiffs, nsdiffs
from pmdarima.model_selection import cross_val_score, RollingForecastCV
from sklearn.preprocessing import StandardScaler
from statsmodels.graphics.tsaplots import plot_ccf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from joblib import Parallel, delayed

In [ ]:
#Fetch data
LAT, LON = 56.1629, 10.2039  # Aarhus
START, END = "2025-04-18", "2026-04-18"

# ── 1. Historical air quality (NO2, O3) ──────────────────────────────────────
print("Fetching air quality...")
aq = requests.get(
    "https://air-quality-api.open-meteo.com/v1/air-quality",
    params={
        "latitude": LAT,
        "longitude": LON,
        "hourly": "nitrogen_dioxide,ozone,pm10,pm2_5",
        "start_date": START,
        "end_date": END,
        "timezone": "Europe/Copenhagen"
    }
).json()

aq_df = pd.DataFrame({
    "Recorded": pd.to_datetime(aq["hourly"]["time"]),
    "NO2":      aq["hourly"]["nitrogen_dioxide"],
    "O3":       aq["hourly"]["ozone"],
    "PM10":     aq["hourly"]["pm10"],
    "PM2.5":    aq["hourly"]["pm2_5"],
})

# ── 2. Historical weather (wind, rain, temp, radiation) ──────────────────────
print("Fetching weather...")
wx = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": LAT,
        "longitude": LON,
        "hourly": "wind_speed_10m,temperature_2m,shortwave_radiation,precipitation,relative_humidity_2m",
        "wind_speed_unit": "ms",
        "start_date": START,
        "end_date": END,
        "timezone": "Europe/Copenhagen"
    }
).json()

wx_df = pd.DataFrame({
    "Recorded":           pd.to_datetime(wx["hourly"]["time"]),
    "wind_speed":         wx["hourly"]["wind_speed_10m"],
    "temperature":        wx["hourly"]["temperature_2m"],
    "solar_radiation":    wx["hourly"]["shortwave_radiation"],
    "precipitation":      wx["hourly"]["precipitation"],
    "humidity":           wx["hourly"]["relative_humidity_2m"],

})

# ── 3. Merge and save ─────────────────────────────────────────────────────────
print("Merging...")
df = aq_df.merge(wx_df, on="Recorded", how="inner")
df = df.sort_values("Recorded").reset_index(drop=True)

print(f"\nFinal shape: {df.shape}")
print(df.head())
print(f"\nDate range: {df.Recorded.min()} → {df.Recorded.max()}")
print(f"Missing values:\n{df.isnull().sum()}")


In [ ]:
# Setup
df["Recorded"] = pd.to_datetime(df["Recorded"])
df = df.set_index('Recorded')


target_col = 'NO2'
m = 24  # seasonal period (hourly data)

y = df[target_col].values
X = df[["temperature"]].values

# train/test split — hold out last 4 days (96 hours)
holdout = 96
y_train, y_test = y[:-holdout], y[-holdout:]
X_train, X_test = X[:-holdout], X[-holdout:]

In [ ]:
#Scale exogenous features:

# Create scaler
X_scaler = StandardScaler()

#IF only one predictor:
X_train = X_train.reshape(-1, 1)
X_test = X_test.reshape(-1, 1)

# Fit on training data only
X_train_scaled = X_scaler.fit_transform(X_train)

# Use same scaling parameters on test data
X_test_scaled = X_scaler.transform(X_test)

In [ ]:
# Determine d and D
d = ndiffs(y_train, test='kpss')
D = nsdiffs(y_train, m=m, test='ch')
print(f"d={d}, D={D}")

In [ ]:
def full_diff(arr, d, D, m):
    # seasonal differencing first
    if arr.ndim == 1:
        for _ in range(D):
            arr = arr[m:] - arr[:-m]
        # then non-seasonal differencing
        for _ in range(d):
            arr = np.diff(arr)
    else:
        for _ in range(D):
            arr = arr[m:] - arr[:-m]
        for _ in range(d):
            arr = np.diff(arr, axis=0)
    return arr

y_train_diff = full_diff(y_train, d=d, D=1, m=24)
X_train_diff = full_diff(X_train_scaled, d=d, D=1, m=24)

In [ ]:
#plot
feature_cols = ["wind_speed", "temperature", "solar_radiation", "precipitation", "humidity"]
target_col = "NO2"

fig, axes = plt.subplots(len(feature_cols), 1, figsize=(12, 3 * len(feature_cols)))

for i, col in enumerate(feature_cols):
    plot_ccf(y_train_diff, X_train_diff[:, i], lags=48, ax=axes[i], negative_lags=False, alpha = 0.05)
    axes[i].set_title(f'CCF (differenced): {target_col} vs {col}')

plt.tight_layout()
plt.show()